# Python File Handling — Combined Practice Projects

---

> **Pre-requisite:** Complete **Notebook 1** (python_file_handling_notes.ipynb) before starting this one.

This notebook has **4 real-world mini projects** that combine Text files, CSV, and JSON together.

## Projects Overview

| Project | Theme | What it combines |
|---------|-------|------------------|
| 1 | Student Records Manager | JSON → CSV → Text Report |
| 2 | Log File Analyzer | Text Log → CSV (errors) → JSON Summary |
| 3 | Shopping Cart System | JSON catalog → CSV orders → Text invoice |
| 4 | Data Format Converter | CSV → JSON → Text (all formats) |

---

## How to use this notebook
- Each project has an explanation, then step-by-step code cells
- Run cells **top to bottom** inside each project
- At the end of each project there is a **mini challenge** for you to try!
- Code is **heavily commented** so you understand every line

> **Tip:** All files created by these projects will appear in the same folder as this notebook.

---
# Common Mistakes and Quick Fixes
---

Before we start, here are the bugs that trip up beginners the most:

| Mistake | Wrong Code | Fixed Code |
|---------|-----------|------------|
| CSV blank lines | `open("f.csv", "w")` | `open("f.csv", "w", newline="")` |
| Overwriting instead of appending | `open("f.txt", "w")` | `open("f.txt", "a")` |
| Single quotes in JSON | `json.loads("{'k': 'v'}")` | Use double quotes inside JSON |
| Missing encoding | `open("f.txt", "r")` | `open("f.txt", "r", encoding="utf-8")` |
| Not handling missing file | `open("f.txt", "r")` | Wrap in `try/except FileNotFoundError` |

We will intentionally show some of these bugs in action during the projects so you learn to spot them!

---
# What Went Wrong? — Spot the Bug Exercises
---

Before the projects, let us look at **intentionally broken code** and understand why it fails.

Run each cell below and read the error carefully. Then look at the fix!

In [ ]:
# BUG 1: Forgetting newline='' in CSV (causes blank lines between rows)

import csv

# BUGGY - missing newline=''
with open('bug_csv.csv', 'w', encoding='utf-8') as f:  # no newline='' !
    writer = csv.writer(f)
    writer.writerow(['Name', 'Age'])
    writer.writerow(['Alice', 25])
    writer.writerow(['Bob', 30])

print('BUGGY version raw content (notice double line endings):')
with open('bug_csv.csv', 'rb') as f:  # rb = read bytes, to see hidden chars
    print(repr(f.read()))

print()

# FIX - add newline=''
with open('bug_csv.csv', 'w', newline='', encoding='utf-8') as f:  # FIXED!
    writer = csv.writer(f)
    writer.writerow(['Name', 'Age'])
    writer.writerow(['Alice', 25])
    writer.writerow(['Bob', 30])

print('FIXED version raw content (single line endings):')
with open('bug_csv.csv', 'rb') as f:
    print(repr(f.read()))

In [ ]:
# BUG 2: Using 'w' mode thinking it appends (it ERASES all existing data!)
import json

# Write initial data
with open('bug_erase.json', 'w', encoding='utf-8') as f:
    json.dump({'name': 'Alice', 'score': 90}, f)

print('Original file:')
with open('bug_erase.json', 'r', encoding='utf-8') as f:
    print(f.read())

# BUGGY - using 'w' when trying to add more data (it ERASES everything!)
with open('bug_erase.json', 'w', encoding='utf-8') as f:
    json.dump({'grade': 'A'}, f)  # Original data is GONE!

print('After BUGGY write (data lost!):')
with open('bug_erase.json', 'r', encoding='utf-8') as f:
    print(f.read())

# FIX - Read first, then modify, then write back
with open('bug_erase.json', 'w', encoding='utf-8') as f:
    json.dump({'name': 'Alice', 'score': 90}, f)  # Reset to original

with open('bug_erase.json', 'r', encoding='utf-8') as f:
    existing = json.load(f)       # Step 1: Read
existing['grade'] = 'A'           # Step 2: Modify
with open('bug_erase.json', 'w', encoding='utf-8') as f:
    json.dump(existing, f, indent=2)  # Step 3: Write back

print('After FIXED update (all data preserved!):')
with open('bug_erase.json', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# BUG 3: Using str() instead of json.dumps() to create a JSON string
import json

python_dict = {'name': 'Alice', 'age': 25}

# BUGGY - str() creates Python syntax, NOT valid JSON!
raw_string = str(python_dict)
print('str() output:', raw_string)

try:
    result = json.loads(raw_string)  # This will FAIL!
    print('Loaded:', result)
except json.JSONDecodeError as e:
    print('Error:', e.msg, '(single quotes are not valid JSON!)')

print()

# FIX - use json.dumps() to create a proper JSON string
json_string = json.dumps(python_dict)  # Proper JSON string
print('json.dumps() output:', json_string)

result = json.loads(json_string)  # Works perfectly!
print('Loaded back:', result)

---
# Project 1: Student Records Manager
---

## Goal
Manage student records across multiple file formats:

```
students.json  --read-->  Python  --write-->  students.csv  --read-->  report.txt
```

## What we will build
- Store student data in a **JSON file**
- Read it and save as a **CSV file**
- Read CSV and generate a formatted **text report**
- Add a new student to JSON and regenerate everything

## Files created
- `p1_students.json`
- `p1_students.csv`
- `p1_report.txt`

In [ ]:
import json, csv

# STEP 1: Create student data and save to JSON

students = [
    {'name': 'Alice',   'age': 21, 'subject': 'Python',       'marks': 88, 'grade': 'B+'},
    {'name': 'Bob',     'age': 22, 'subject': 'Data Science', 'marks': 72, 'grade': 'B'},
    {'name': 'Charlie', 'age': 20, 'subject': 'Python',       'marks': 95, 'grade': 'A+'},
    {'name': 'Diana',   'age': 23, 'subject': 'SQL',          'marks': 60, 'grade': 'C+'},
    {'name': 'Evan',    'age': 21, 'subject': 'Data Science', 'marks': 45, 'grade': 'D'},
]

with open('p1_students.json', 'w', encoding='utf-8') as f:
    json.dump(students, f, indent=4)

print('Step 1 Done: Saved', len(students), 'students to p1_students.json')
print()

# Verify by reading back
with open('p1_students.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('Verification - first student from JSON:')
print(json.dumps(loaded[0], indent=2))

In [ ]:
# STEP 2: Read JSON and convert to CSV

with open('p1_students.json', 'r', encoding='utf-8') as f:
    students = json.load(f)

# Write to CSV using DictWriter (keys become column headers automatically)
with open('p1_students.csv', 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['name', 'age', 'subject', 'marks', 'grade']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()       # Writes: name,age,subject,marks,grade
    writer.writerows(students) # Writes all student rows from list of dicts

print('Step 2 Done: JSON converted to p1_students.csv')
print()
print('CSV file content:')
print('-' * 50)
with open('p1_students.csv', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 3: Read CSV and generate a formatted text report

with open('p1_students.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)       # Read as dicts
    students = list(reader)          # Convert to list so we can use multiple times

# Calculate statistics
total       = len(students)
all_marks   = [int(s['marks']) for s in students]
avg_marks   = sum(all_marks) / total
top_student = max(students, key=lambda s: int(s['marks']))
passed      = [s for s in students if int(s['marks']) >= 50]
failed      = [s for s in students if int(s['marks']) < 50]

# Build the report as a list of lines
report_lines = [
    '=' * 50 + '\n',
    '       STUDENT PERFORMANCE REPORT\n',
    '=' * 50 + '\n\n',
    f'Total Students : {total}\n',
    f'Average Marks  : {avg_marks:.2f}\n',
    f'Top Student    : {top_student["name"]} ({top_student["marks"]} marks)\n',
    f'Passed         : {len(passed)}\n',
    f'Failed         : {len(failed)}\n\n',
    '-' * 50 + '\n',
    'Detailed Results:\n',
    '-' * 50 + '\n',
]

for s in students:
    status = 'PASS' if int(s['marks']) >= 50 else 'FAIL'
    line = f"{s['name']:<12} | {s['subject']:<15} | Marks: {s['marks']:<4} | Grade: {s['grade']:<3} | {status}\n"
    report_lines.append(line)

report_lines.append('=' * 50 + '\n')

# Write report to text file
with open('p1_report.txt', 'w', encoding='utf-8') as f:
    f.writelines(report_lines)

print('Step 3 Done: Text report saved to p1_report.txt')
print()
with open('p1_report.txt', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 4: Add a new student to JSON and regenerate CSV + report

new_student = {'name': 'Fatima', 'age': 22, 'subject': 'Python', 'marks': 91, 'grade': 'A'}

# Read existing JSON
with open('p1_students.json', 'r', encoding='utf-8') as f:
    students = json.load(f)

students.append(new_student)  # Add new student

# Save updated list back to JSON
with open('p1_students.json', 'w', encoding='utf-8') as f:
    json.dump(students, f, indent=4)

print(f'Added {new_student["name"]} to JSON. Total now: {len(students)}')

# Regenerate CSV
with open('p1_students.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'age', 'subject', 'marks', 'grade'])
    writer.writeheader()
    writer.writerows(students)
print('CSV regenerated!')

# Regenerate report
with open('p1_students.csv', 'r', encoding='utf-8') as f:
    students = list(csv.DictReader(f))

total  = len(students)
avg    = sum(int(s['marks']) for s in students) / total
top    = max(students, key=lambda s: int(s['marks']))
passed = [s for s in students if int(s['marks']) >= 50]
failed = [s for s in students if int(s['marks']) < 50]

lines = [
    '=' * 50 + '\n',
    '       UPDATED STUDENT PERFORMANCE REPORT\n',
    '=' * 50 + '\n\n',
    f'Total Students : {total}\n',
    f'Average Marks  : {avg:.2f}\n',
    f'Top Student    : {top["name"]} ({top["marks"]} marks)\n',
    f'Passed: {len(passed)}  |  Failed: {len(failed)}\n\n',
    '-' * 50 + '\n',
    'Detailed Results:\n',
    '-' * 50 + '\n',
]
for s in students:
    status = 'PASS' if int(s['marks']) >= 50 else 'FAIL'
    lines.append(f"{s['name']:<12} | {s['subject']:<15} | {s['marks']:<4} | {s['grade']:<3} | {status}\n")
lines.append('=' * 50 + '\n')

with open('p1_report.txt', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print('Report regenerated!')
print()
with open('p1_report.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## Your Challenge — Project 1

**Easy:**
1. Add 2 more students of your choice to `p1_students.json` and regenerate the CSV and report.
2. Change the pass mark from 50 to 60 in Step 3 and see how the report changes.

**Medium:**
3. Add a **Subject-wise Summary** section to the text report showing average marks per subject.

**Expected output for medium challenge:**
```
Subject-wise Summary:
  Python       : Avg = 91.33
  Data Science : Avg = 58.50
  SQL          : Avg = 60.00
```

> Hint: Use a dictionary to group students by subject, then calculate average for each group.

---
# Project 2: Log File Analyzer
---

## Goal
Process a log file and extract useful information into structured formats:

```
app.log (Text)  --parse-->  errors.csv (CSV)  --summarize-->  summary.json (JSON)
```

## What we will build
- Generate a sample **log text file** with timestamped entries
- Parse the log and extract only **ERROR lines** into a CSV file
- Count errors by type and save the **summary as JSON**
- Read JSON summary and print a clean report

## Files created
- `p2_app.log`
- `p2_errors.csv`
- `p2_summary.json`

In [ ]:
import json, csv

# STEP 1: Generate a sample log file (simulating a real app)

log_entries = [
    '2024-01-15 08:00:01 INFO  Application started successfully\n',
    '2024-01-15 08:01:12 INFO  User Alice logged in\n',
    '2024-01-15 08:02:30 ERROR DatabaseError: Connection timeout after 30s\n',
    '2024-01-15 08:03:05 INFO  User Bob logged in\n',
    '2024-01-15 08:04:22 WARNING High memory usage: 85 percent\n',
    '2024-01-15 08:05:11 ERROR FileNotFoundError: config.yaml not found\n',
    '2024-01-15 08:06:33 INFO  Scheduled backup started\n',
    '2024-01-15 08:07:44 ERROR DatabaseError: Query took too long\n',
    '2024-01-15 08:08:55 INFO  Backup completed successfully\n',
    '2024-01-15 08:09:10 ERROR NetworkError: API endpoint unreachable\n',
    '2024-01-15 08:10:20 WARNING Disk space below 20 percent\n',
    '2024-01-15 08:11:30 ERROR FileNotFoundError: template.html not found\n',
    '2024-01-15 08:12:45 INFO  User Charlie logged in\n',
    '2024-01-15 08:13:50 ERROR DatabaseError: Deadlock detected\n',
    '2024-01-15 08:14:00 INFO  Application shutdown initiated\n',
]

with open('p2_app.log', 'w', encoding='utf-8') as f:
    f.writelines(log_entries)

print('Step 1 Done: Generated p2_app.log with', len(log_entries), 'entries')
print()
print('Log file content:')
print('-' * 60)
with open('p2_app.log', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 2: Parse log file and extract ERROR lines into CSV

errors_found = []

with open('p2_app.log', 'r', encoding='utf-8') as f:
    for line in f:              # Read line by line (memory efficient!)
        line = line.strip()     # Remove leading/trailing whitespace

        if 'ERROR' in line:     # Only process error lines
            # Log format: YYYY-MM-DD HH:MM:SS LEVEL  Message
            parts = line.split(' ', 3)   # Split into max 4 parts
            # parts[0]=date, parts[1]=time, parts[2]=level, parts[3]=message

            date_part  = parts[0]              # '2024-01-15'
            time_part  = parts[1]              # '08:02:30'
            message    = parts[3]              # 'DatabaseError: Connection...'
            error_type = message.split(':')[0] # 'DatabaseError'

            errors_found.append({
                'date':       date_part,
                'time':       time_part,
                'error_type': error_type,
                'message':    message
            })

# Write errors to CSV
with open('p2_errors.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['date', 'time', 'error_type', 'message'])
    writer.writeheader()
    writer.writerows(errors_found)

print(f'Step 2 Done: Found {len(errors_found)} errors, saved to p2_errors.csv')
print()
print('Errors CSV:')
print('-' * 60)
with open('p2_errors.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['time']} | {row['error_type']:<22} | {row['message']}")

In [ ]:
# STEP 3: Count errors by type and save summary to JSON

with open('p2_errors.csv', 'r', encoding='utf-8') as f:
    errors = list(csv.DictReader(f))

# Count how many times each error type appeared
error_counts = {}
for error in errors:
    etype = error['error_type']
    if etype in error_counts:
        error_counts[etype] += 1   # Increment if already seen
    else:
        error_counts[etype] = 1    # First time seeing this type

# Build summary dictionary
summary = {
    'log_file':          'p2_app.log',
    'total_errors':      len(errors),
    'error_breakdown':   error_counts,
    'most_common_error': max(error_counts, key=error_counts.get)
}

with open('p2_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4)

print('Step 3 Done: Summary saved to p2_summary.json')
print()
print('JSON summary content:')
with open('p2_summary.json', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 4: Read JSON summary and print a clean report

with open('p2_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('=' * 45)
print('         LOG ANALYSIS REPORT')
print('=' * 45)
print(f"Log File      : {summary['log_file']}")
print(f"Total Errors  : {summary['total_errors']}")
print(f"Most Common   : {summary['most_common_error']}")
print()
print('Error Breakdown:')
print('-' * 45)
for error_type, count in summary['error_breakdown'].items():
    bar = '#' * count  # Simple text bar chart!
    print(f'{error_type:<25} : {count} {bar}')
print('=' * 45)

## Your Challenge — Project 2

**Easy:**
1. Add 3 more log entries to `p2_app.log` (2 ERRORs and 1 WARNING) manually, then re-run Steps 2 to 4 and see the updated report.

**Medium:**
2. Add a `warnings_count` key to the JSON summary that counts how many WARNING lines were in the log.

**Expected output:**
```json
{
    "log_file": "p2_app.log",
    "total_errors": 5,
    "warnings_count": 2,
    ...
}
```

> Hint: Add an `if 'WARNING' in line` check in Step 2 and count those separately.

---
# Project 3: Shopping Cart System
---

## Goal
Simulate a simple shopping system:

```
products.json  --read-->  add to cart  --write-->  orders.csv  --read-->  invoice.txt
```

## What we will build
- Store product catalog in a **JSON file**
- Simulate purchases and save order history to **CSV**
- Read orders and generate a formatted **invoice text file**

## Files created
- `p3_products.json`
- `p3_orders.csv`
- `p3_invoice.txt`

In [ ]:
import json, csv

# STEP 1: Create product catalog and save to JSON

products = [
    {'id': 'P001', 'name': 'Python Book',      'price': 499,  'category': 'Books'},
    {'id': 'P002', 'name': 'USB-C Cable',      'price': 149,  'category': 'Electronics'},
    {'id': 'P003', 'name': 'Notebook (A4)',    'price': 60,   'category': 'Stationery'},
    {'id': 'P004', 'name': 'Wireless Mouse',   'price': 899,  'category': 'Electronics'},
    {'id': 'P005', 'name': 'Sticky Notes',     'price': 45,   'category': 'Stationery'},
    {'id': 'P006', 'name': 'Data Science Book','price': 599,  'category': 'Books'},
]

with open('p3_products.json', 'w', encoding='utf-8') as f:
    json.dump(products, f, indent=4)

print('Step 1 Done: Product catalog saved to p3_products.json')
print()
print('Available Products:')
print('-' * 50)
for p in products:
    print(f"[{p['id']}] {p['name']:<22} Rs.{p['price']:<6} ({p['category']})")

In [ ]:
# STEP 2: Simulate purchases and save to CSV

# Load product catalog from JSON
with open('p3_products.json', 'r', encoding='utf-8') as f:
    catalog = json.load(f)

# Build a lookup dictionary: product_id -> product dict
# This lets us find a product instantly instead of looping every time
product_lookup = {p['id']: p for p in catalog}

# Simulated customer cart: list of (product_id, quantity) tuples
cart = [
    ('P001', 2),   # 2 Python Books
    ('P003', 5),   # 5 Notebooks
    ('P004', 1),   # 1 Wireless Mouse
    ('P005', 3),   # 3 Sticky Notes packs
    ('P002', 1),   # 1 USB-C Cable
]

# Process cart and build order rows
order_rows = []
for product_id, qty in cart:
    product     = product_lookup[product_id]  # Fetch product details
    unit_price  = product['price']
    total_price = unit_price * qty            # Calculate line total

    order_rows.append({
        'product_id':   product_id,
        'product_name': product['name'],
        'category':     product['category'],
        'quantity':     qty,
        'unit_price':   unit_price,
        'total_price':  total_price
    })

# Save orders to CSV
with open('p3_orders.csv', 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['product_id', 'product_name', 'category', 'quantity', 'unit_price', 'total_price']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(order_rows)

print('Step 2 Done: Orders saved to p3_orders.csv')
print()
print('Orders CSV:')
print('-' * 70)
with open('p3_orders.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['product_name']:<22} | Qty: {row['quantity']} | Unit: Rs.{row['unit_price']:<6} | Total: Rs.{row['total_price']}")

In [ ]:
# STEP 3: Read orders and generate formatted invoice text file

with open('p3_orders.csv', 'r', encoding='utf-8') as f:
    orders = list(csv.DictReader(f))

# Calculate totals
grand_total = sum(int(o['total_price']) for o in orders)
total_items = sum(int(o['quantity']) for o in orders)

# Build invoice as a list of lines
invoice = []
invoice.append('=' * 55 + '\n')
invoice.append('           SHOPPING INVOICE\n')
invoice.append('=' * 55 + '\n')
invoice.append('Customer: Valued Customer\n')
invoice.append('Date: 2024-01-15\n')
invoice.append('-' * 55 + '\n')
invoice.append(f"{'Item':<22} {'Qty':>4} {'Unit Price':>12} {'Total':>10}\n")
invoice.append('-' * 55 + '\n')

for o in orders:
    line = f"{o['product_name']:<22} {o['quantity']:>4} {'Rs.'+o['unit_price']:>12} {'Rs.'+o['total_price']:>10}\n"
    invoice.append(line)

invoice.append('-' * 55 + '\n')
invoice.append(f'Total Items Ordered : {total_items}\n')
invoice.append(f'GRAND TOTAL         : Rs.{grand_total}\n')
invoice.append('=' * 55 + '\n')
invoice.append('Thank you for shopping with us!\n')

with open('p3_invoice.txt', 'w', encoding='utf-8') as f:
    f.writelines(invoice)

print('Step 3 Done: Invoice saved to p3_invoice.txt')
print()
with open('p3_invoice.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## Your Challenge — Project 3

**Easy:**
1. Add 2 more products to `p3_products.json` and add them to the cart with a quantity of your choice. Regenerate the invoice.

**Medium:**
2. Add a **Category Summary** section at the bottom of the invoice showing total spent per category.

**Expected output:**
```
Category Summary:
  Books        : Rs.1598
  Stationery   : Rs.435
  Electronics  : Rs.1048
```

> Hint: Group orders by category using a dictionary, then add the totals.

---
# Project 4: Data Format Converter
---

## Goal
Build a **reusable utility** that converts data between all three formats:

```
CSV  --convert-->  JSON
JSON --convert-->  CSV
CSV  --convert-->  Formatted Text Table
JSON --convert-->  Formatted Text Table
```

## What we will build
- A CSV file of contacts as starting data
- Reusable converter functions for all format combinations
- Demo: use the converter on our contacts AND on the student data from Project 1

## Files created
- `p4_contacts.csv`
- `p4_contacts.json`
- `p4_contacts.txt`

In [ ]:
import json, csv

# STEP 1: Create a contacts CSV file (our starting point)

contacts_data = [
    ['Name',    'Phone',       'Email',              'City'],
    ['Alice',   '9876543210',  'alice@email.com',    'Mumbai'],
    ['Bob',     '8765432109',  'bob@email.com',      'Delhi'],
    ['Charlie', '7654321098',  'charlie@email.com',  'Bangalore'],
    ['Diana',   '6543210987',  'diana@email.com',    'Chennai'],
    ['Evan',    '5432109876',  'evan@email.com',     'Kolkata'],
]

with open('p4_contacts.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(contacts_data)

print('Step 1 Done: p4_contacts.csv created')
print()
print('CSV content:')
with open('p4_contacts.csv', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 2: Define all converter functions
# These are reusable - you can use them on ANY csv or json file!

def csv_to_json(csv_file, json_file, indent=4):
    """
    Convert a CSV file to a JSON file.
    First row of CSV is used as the dictionary keys (column headers).
    Returns the loaded data as a list of dicts.
    """
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)   # Reads rows as dicts using header as keys
        data = list(reader)          # Convert to list of dicts

    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=indent)

    print(f'Converted: {csv_file} --> {json_file}  ({len(data)} records)')
    return data


def json_to_csv(json_file, csv_file):
    """
    Convert a JSON file (list of dicts) to a CSV file.
    JSON keys become column headers automatically.
    """
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print('JSON file is empty!')
        return

    fieldnames = list(data[0].keys())  # Get column names from first dict's keys

    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

    print(f'Converted: {json_file} --> {csv_file}  ({len(data)} records)')


def csv_to_text(csv_file, txt_file, title='Data Report'):
    """
    Convert a CSV file to a neatly formatted text table.
    Column widths are auto-calculated to fit the longest value.
    """
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader  = csv.DictReader(f)
        rows    = list(reader)
        headers = reader.fieldnames

    # Calculate how wide each column needs to be
    col_widths = {h: len(h) for h in headers}
    for row in rows:
        for h in headers:
            col_widths[h] = max(col_widths[h], len(str(row[h])))

    # Build separator line: +-------+-------+
    sep = '+' + '+'.join('-' * (col_widths[h] + 2) for h in headers) + '+\n'

    # Build header row: | Name  | Age  |
    header_row = '|' + '|'.join(f" {h:<{col_widths[h]}} " for h in headers) + '|\n'

    lines = []
    lines.append(f'{title}\n')
    lines.append('=' * len(sep.strip()) + '\n')
    lines.append(sep)
    lines.append(header_row)
    lines.append(sep)
    for row in rows:
        data_row = '|' + '|'.join(f" {str(row[h]):<{col_widths[h]}} " for h in headers) + '|\n'
        lines.append(data_row)
    lines.append(sep)
    lines.append(f'Total records: {len(rows)}\n')

    with open(txt_file, 'w', encoding='utf-8') as f:
        f.writelines(lines)

    print(f'Converted: {csv_file} --> {txt_file}  (formatted table)')


print('All converter functions defined!')

In [ ]:
# STEP 3: Run all converters on our contacts data

print('Running all conversions on contacts...')
print('=' * 50)

# CSV -> JSON
csv_to_json('p4_contacts.csv', 'p4_contacts.json')

# JSON -> CSV (round-trip test: JSON back to a new CSV)
json_to_csv('p4_contacts.json', 'p4_contacts_from_json.csv')

# CSV -> Formatted Text Table
csv_to_text('p4_contacts.csv', 'p4_contacts.txt', title='Contacts Directory')

print()
print('All conversions done! Viewing results...')
print()

print('JSON output:')
print('-' * 50)
with open('p4_contacts.json', 'r', encoding='utf-8') as f:
    print(f.read())

print('Text table output:')
print('-' * 50)
with open('p4_contacts.txt', 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# STEP 4: Use the same converters on the student data from Project 1!
# This proves the functions are truly reusable.

print('Converting student data from Project 1:')
print('=' * 50)

try:
    # Convert students CSV to JSON
    students = csv_to_json('p1_students.csv', 'p1_students_converted.json')
    print()
    print('First student from converted JSON:')
    print(json.dumps(students[0], indent=2))
    print()

    # Convert students CSV to text table
    csv_to_text('p1_students.csv', 'p1_students_table.txt', title='Student Results Table')
    print()

    print('Student text table:')
    with open('p1_students_table.txt', 'r', encoding='utf-8') as f:
        print(f.read())

except FileNotFoundError:
    print('p1_students.csv not found. Please run Project 1 first!')

## Your Challenge — Project 4

**Easy:**
1. Add 3 more contacts to `p4_contacts.csv` and re-run the converters. See all 3 output files update automatically.

**Medium:**
2. Write a new `json_to_text()` function that converts a JSON file directly to a formatted text table (without going through CSV).

**Function signature to implement:**
```python
def json_to_text(json_file, txt_file, title='Data Report'):
    # Step 1: Read JSON into a list of dicts
    # Step 2: Get headers from the keys of the first dict
    # Step 3: Build the text table (same logic as csv_to_text)
    # Step 4: Write to txt_file
    pass
```

> Hint: The table-building logic is identical to `csv_to_text()`. The only difference is how you load the data (json.load instead of csv.DictReader).

---
# Well Done! You have completed both notebooks!
---

## Skills you practised in this notebook

| Skill | Where you used it |
|-------|------------------|
| Read and Write JSON files | All 4 projects |
| Read and Write CSV files | All 4 projects |
| Read and Write Text files | Projects 1, 2, 3 |
| Append to files | Project 1 (adding new student) |
| Loop through file line by line | Project 2 (log parser) |
| try/except for file errors | Projects 2, 4 |
| Converting between formats | Project 4 |
| Building reusable functions | Project 4 |
| Using DictReader and DictWriter | Projects 1, 2, 3, 4 |

---

## All Files Created by this Notebook

| File | Created By |
|------|------------|
| `p1_students.json` | Project 1 |
| `p1_students.csv` | Project 1 |
| `p1_report.txt` | Project 1 |
| `p2_app.log` | Project 2 |
| `p2_errors.csv` | Project 2 |
| `p2_summary.json` | Project 2 |
| `p3_products.json` | Project 3 |
| `p3_orders.csv` | Project 3 |
| `p3_invoice.txt` | Project 3 |
| `p4_contacts.csv` | Project 4 |
| `p4_contacts.json` | Project 4 |
| `p4_contacts.txt` | Project 4 |

---

## What to Learn Next

| Topic | Why | How to start |
|-------|-----|----------|
| **pandas** | Read/write CSV and Excel with 1 line of code | `pip install pandas` |
| **requests** | Fetch real JSON data from live APIs | `pip install requests` |
| **sqlite3** | Store data in a proper database | Built-in Python module |
| **os.walk** | Process entire folders of files | Built-in Python module |
| **pickle** | Save any Python object to a file | Built-in Python module |

---

> Keep building small projects! Pick something you care about and build it. Good luck!